In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv('data/application_train.csv')
df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(41), str(16)
memory usage: 286.2 MB


In [4]:
share = df['TARGET'].value_counts(normalize=True)
print(share)

TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [5]:
missing_series = df.isnull().mean().nlargest(5)
print(missing_series)

COMMONAREA_AVG              0.698723
COMMONAREA_MODE             0.698723
COMMONAREA_MEDI             0.698723
NONLIVINGAPARTMENTS_AVG     0.694330
NONLIVINGAPARTMENTS_MODE    0.694330
dtype: float64


In [6]:
df['AGE'] = df['DAYS_BIRTH'] / -365

/var/folders/tg/m3kj7c9d5t9gsv408k8gwqkm0000gn/T/ipykernel_2514/468626725.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['AGE'] = df['DAYS_BIRTH'] / -365


In [7]:
df['AGE']

0         25.920548
1         45.931507
2         52.180822
3         52.068493
4         54.608219
            ...    
307506    25.553425
307507    56.917808
307508    41.002740
307509    32.769863
307510    46.180822
Name: AGE, Length: 307511, dtype: float64

In [8]:
mean_age_by_target = df.groupby('TARGET')['AGE'].mean()
print(mean_age_by_target)

TARGET
0    44.214182
1    40.780351
Name: AGE, dtype: float64


In [9]:
df[[col for col in df.columns if col.startswith('DAYS_')]].describe()

,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,DAYS_LAST_PHONE_CHANGE
count,307511.000000,307511.000000,307511.000000,307511.000000,307510.000000
mean,-16036.995067,63815.045904,-4986.120328,-2994.202373,-962.858788
std,4363.988632,141275.766519,3522.886321,1509.450419,826.808487
min,-25229.000000,-17912.000000,-24672.000000,-7197.000000,-4292.000000
25%,-19682.000000,-2760.000000,-7479.500000,-4299.000000,-1570.000000
50%,-15750.000000,-1213.000000,-4504.000000,-3254.000000,-757.000000
75%,-12413.000000,-289.000000,-2010.000000,-1720.000000,-274.000000
max,-7489.000000,365243.000000,0.000000,0.000000,0.000000


In [10]:
df.filter(regex='^DAYS').describe()

,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,DAYS_LAST_PHONE_CHANGE
count,307511.000000,307511.000000,307511.000000,307511.000000,307510.000000
mean,-16036.995067,63815.045904,-4986.120328,-2994.202373,-962.858788
std,4363.988632,141275.766519,3522.886321,1509.450419,826.808487
min,-25229.000000,-17912.000000,-24672.000000,-7197.000000,-4292.000000
25%,-19682.000000,-2760.000000,-7479.500000,-4299.000000,-1570.000000
50%,-15750.000000,-1213.000000,-4504.000000,-3254.000000,-757.000000
75%,-12413.000000,-289.000000,-2010.000000,-1720.000000,-274.000000
max,-7489.000000,365243.000000,0.000000,0.000000,0.000000


In [11]:
days_cols = [c for c in df.columns if c.startswith("DAYS_")]
df[days_cols].describe().T[["min", "max"]]

,min,max
DAYS_BIRTH,-25229.0,-7489.0
DAYS_EMPLOYED,-17912.0,365243.0
DAYS_REGISTRATION,-24672.0,0.0
DAYS_ID_PUBLISH,-7197.0,0.0
DAYS_LAST_PHONE_CHANGE,-4292.0,0.0


In [12]:
anom = df["DAYS_EMPLOYED"] == 365243

In [13]:
print(anom.sum(), anom.mean())

55374 0.18007160719453938


In [14]:
print(df.loc[anom, "NAME_INCOME_TYPE"].value_counts(), "\n")
print(df.loc[~anom, "NAME_INCOME_TYPE"].value_counts())

NAME_INCOME_TYPE
Pensioner     55352
Unemployed       22
Name: count, dtype: int64 

NAME_INCOME_TYPE
Working                 158774
Commercial associate     71617
State servant            21703
Student                     18
Pensioner                   10
Businessman                 10
Maternity leave              5
Name: count, dtype: int64


In [15]:
print(df.loc[anom, "TARGET"].mean(), df.loc[~anom, "TARGET"].mean())

0.05399646043269404 0.0865997453765215


In [16]:
df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

/var/folders/tg/m3kj7c9d5t9gsv408k8gwqkm0000gn/T/ipykernel_2514/4082870198.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)


In [17]:
from sklearn.model_selection import train_test_split

y = df["TARGET"]
X = df.drop(columns=["TARGET", "SK_ID_CURR"])
cat_cols = X.select_dtypes("object").columns
X[cat_cols] = X[cat_cols].astype("category")

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

/var/folders/tg/m3kj7c9d5t9gsv408k8gwqkm0000gn/T/ipykernel_2514/2812705785.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes("object").columns


In [18]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="median")),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000))
])

In [19]:
X_tr_d = pd.get_dummies(X_tr, dtype=float)
X_va_d = pd.get_dummies(X_va, dtype=float)

X_va_d = X_va_d.reindex(columns=X_tr_d.columns, fill_value=0)

In [20]:
from sklearn.metrics import roc_auc_score

pipe.fit(X_tr_d, y_tr)

p_va = pipe.predict_proba(X_va_d)[:, 1]
auc = roc_auc_score(y_va, p_va)
print(f"AUC: {auc:.4f}, Gini: {2 * auc - 1:.4f}")

AUC: 0.7491, Gini: 0.4982


In [21]:
import lightgbm as lgb

model = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, verbose=-1)
model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="auc", callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)])

/Users/daniil/projects/credit-scoring/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[200]	valid_0's auc: 0.759252	valid_0's binary_logloss: 0.245458
Early stopping, best iteration is:
[180]	valid_0's auc: 0.759359	valid_0's binary_logloss: 0.245464


,learning_rate,0.05
,n_estimators,2000
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001


In [22]:
lgb_va = model.predict_proba(X_va)[:, 1]
auc = roc_auc_score(y_va, lgb_va)
print(f"AUC: {auc:.4f}, Gini: {2 * auc - 1:.4f}")

AUC: 0.7594, Gini: 0.5187


In [23]:
pd.Series(model.feature_importances_, index=X_tr.columns).sort_values(ascending=False).head(10)

ORGANIZATION_TYPE    800
EXT_SOURCE_1         419
EXT_SOURCE_3         417
EXT_SOURCE_2         353
AMT_CREDIT           265
DAYS_EMPLOYED        208
AMT_ANNUITY          200
AMT_GOODS_PRICE      186
DAYS_ID_PUBLISH      176
DAYS_BIRTH           148
dtype: int32

In [24]:
imp = pd.Series(model.booster_.feature_importance(importance_type="gain"), index=X_tr.columns).sort_values(ascending=False)
imp.head(10)

EXT_SOURCE_3         64252.558055
EXT_SOURCE_2         52047.391343
EXT_SOURCE_1         19100.407058
ORGANIZATION_TYPE    18442.933802
DAYS_EMPLOYED         8186.214707
AMT_CREDIT            7532.314758
AMT_GOODS_PRICE       6183.700101
AMT_ANNUITY           5298.052488
OCCUPATION_TYPE       4496.479593
OWN_CAR_AGE           4128.906419
dtype: float64

In [25]:
bureau = pd.read_csv('data/bureau.csv')
bureau.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [26]:
bureau.info()

<class 'pandas.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 17 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_CURR              int64  
 1   SK_ID_BUREAU            int64  
 2   CREDIT_ACTIVE           str    
 3   CREDIT_CURRENCY         str    
 4   DAYS_CREDIT             int64  
 5   CREDIT_DAY_OVERDUE      int64  
 6   DAYS_CREDIT_ENDDATE     float64
 7   DAYS_ENDDATE_FACT       float64
 8   AMT_CREDIT_MAX_OVERDUE  float64
 9   CNT_CREDIT_PROLONG      int64  
 10  AMT_CREDIT_SUM          float64
 11  AMT_CREDIT_SUM_DEBT     float64
 12  AMT_CREDIT_SUM_LIMIT    float64
 13  AMT_CREDIT_SUM_OVERDUE  float64
 14  CREDIT_TYPE             str    
 15  DAYS_CREDIT_UPDATE      int64  
 16  AMT_ANNUITY             float64
dtypes: float64(8), int64(6), str(3)
memory usage: 222.6 MB


In [27]:
bureau['SK_ID_CURR'].nunique()

305811

In [28]:
in_bureau = df['SK_ID_CURR'].isin(bureau['SK_ID_CURR'])
print(in_bureau.mean())

0.8568506492450677


In [29]:
print(df.groupby(in_bureau)['TARGET'].mean())

SK_ID_CURR
False    0.101249
True     0.077301
Name: TARGET, dtype: float64


In [30]:
bureau["IS_ACTIVE"] = (bureau["CREDIT_ACTIVE"] == "Active").astype(int)
bureau["HAS_OVERDUE"] = (bureau["CREDIT_DAY_OVERDUE"] > 0).astype(int)

agg = bureau.groupby("SK_ID_CURR").agg(
    BUREAU_COUNT=("SK_ID_BUREAU", "count"),
    BUREAU_ACTIVE_COUNT=("IS_ACTIVE", "sum"),
    BUREAU_DAYS_CREDIT_MAX=("DAYS_CREDIT", "max"),
    BUREAU_DAYS_CREDIT_MIN=("DAYS_CREDIT", "min"),
    BUREAU_CREDIT_SUM=("AMT_CREDIT_SUM", "sum"),
    BUREAU_DEBT_SUM=("AMT_CREDIT_SUM_DEBT", "sum"),
    BUREAU_OVERDUE_DAYS_MAX=("CREDIT_DAY_OVERDUE", "max"),
    BUREAU_OVERDUE_AMT_SUM=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    BUREAU_OVERDUE_COUNT=("HAS_OVERDUE", "sum"),
).reset_index()

agg["BUREAU_DEBT_RATIO"] = agg["BUREAU_DEBT_SUM"] / agg["BUREAU_CREDIT_SUM"]
agg["BUREAU_DEBT_RATIO"] = agg["BUREAU_DEBT_RATIO"].replace([np.inf, -np.inf], np.nan)

In [31]:
n_before = len(df)
df2 = df.merge(agg, on="SK_ID_CURR", how="left")
assert len(df2) == n_before, "строки сдвоились или потерялись!"

In [32]:
df2.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,BUREAU_COUNT,BUREAU_ACTIVE_COUNT,BUREAU_DAYS_CREDIT_MAX,BUREAU_DAYS_CREDIT_MIN,BUREAU_CREDIT_SUM,BUREAU_DEBT_SUM,BUREAU_OVERDUE_DAYS_MAX,BUREAU_OVERDUE_AMT_SUM,BUREAU_OVERDUE_COUNT,BUREAU_DEBT_RATIO
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,8.0,2.0,-103.0,-1437.0,865055.565,245781.0,0.0,0.0,0.0,0.284122
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,4.0,1.0,-606.0,-2586.0,1017400.500,0.0,0.0,0.0,0.0,0.000000
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,2.0,0.0,-408.0,-1326.0,189037.800,0.0,0.0,0.0,0.0,0.000000
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,1.0,0.0,-1149.0,-1149.0,146250.000,0.0,0.0,0.0,0.0,0.000000


In [33]:
count_cols = ["BUREAU_COUNT", "BUREAU_ACTIVE_COUNT", "BUREAU_OVERDUE_COUNT"]
df2[count_cols] = df2[count_cols].fillna(0)

In [34]:
y = df2["TARGET"]
X = df2.drop(columns=["TARGET", "SK_ID_CURR"])
cat_cols = X.select_dtypes("object").columns
X[cat_cols] = X[cat_cols].astype("category")

/var/folders/tg/m3kj7c9d5t9gsv408k8gwqkm0000gn/T/ipykernel_2514/4161308274.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes("object").columns


In [35]:
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, verbose=-1)
model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="auc", callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)])

auc = roc_auc_score(y_va, model.predict_proba(X_va)[:, 1])
print(f"AUC = {auc:.4f}, Gini = {2 * auc - 1:.4f}")

/Users/daniil/projects/credit-scoring/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[200]	valid_0's auc: 0.763226	valid_0's binary_logloss: 0.244137
[400]	valid_0's auc: 0.76424	valid_0's binary_logloss: 0.243855
Early stopping, best iteration is:
[474]	valid_0's auc: 0.764521	valid_0's binary_logloss: 0.243811
AUC = 0.7645, Gini = 0.5290


In [36]:
imp = pd.Series(model.booster_.feature_importance(importance_type="gain"), index=X_tr.columns).sort_values(ascending=False)

imp.head(15)

EXT_SOURCE_3              65764.883155
EXT_SOURCE_2              55337.094022
ORGANIZATION_TYPE         25203.353702
EXT_SOURCE_1              21792.573658
DAYS_EMPLOYED             10989.712458
BUREAU_DEBT_RATIO          9806.780395
AMT_CREDIT                 8712.234339
AMT_ANNUITY                8028.203957
BUREAU_DAYS_CREDIT_MAX     7917.579703
AMT_GOODS_PRICE            7795.151139
DAYS_ID_PUBLISH            5693.138043
OWN_CAR_AGE                5331.680917
BUREAU_CREDIT_SUM          5321.724229
OCCUPATION_TYPE            5307.519916
DAYS_BIRTH                 5120.030562
dtype: float64

In [37]:
print(df2.shape)
print([c for c in X_tr.columns if c.startswith("BUREAU")])

(307511, 134)
['BUREAU_COUNT', 'BUREAU_ACTIVE_COUNT', 'BUREAU_DAYS_CREDIT_MAX', 'BUREAU_DAYS_CREDIT_MIN', 'BUREAU_CREDIT_SUM', 'BUREAU_DEBT_SUM', 'BUREAU_OVERDUE_DAYS_MAX', 'BUREAU_OVERDUE_AMT_SUM', 'BUREAU_OVERDUE_COUNT', 'BUREAU_DEBT_RATIO']


In [38]:
df_prev_app = pd.read_csv('data/previous_application.csv')
df_prev_app.head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
df_prev_app.info()

<class 'pandas.DataFrame'>
RangeIndex: 1670214 entries, 0 to 1670213
Data columns (total 37 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   SK_ID_PREV                   1670214 non-null  int64  
 1   SK_ID_CURR                   1670214 non-null  int64  
 2   NAME_CONTRACT_TYPE           1670214 non-null  str    
 3   AMT_ANNUITY                  1297979 non-null  float64
 4   AMT_APPLICATION              1670214 non-null  float64
 5   AMT_CREDIT                   1670213 non-null  float64
 6   AMT_DOWN_PAYMENT             774370 non-null   float64
 7   AMT_GOODS_PRICE              1284699 non-null  float64
 8   WEEKDAY_APPR_PROCESS_START   1670214 non-null  str    
 9   HOUR_APPR_PROCESS_START      1670214 non-null  int64  
 10  FLAG_LAST_APPL_PER_CONTRACT  1670214 non-null  str    
 11  NFLAG_LAST_APPL_IN_DAY       1670214 non-null  int64  
 12  RATE_DOWN_PAYMENT            774370 non-null   float6

In [40]:
print(df_prev_app["NAME_CONTRACT_STATUS"].value_counts())

NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64


In [41]:
df_prev_app["IS_REFUSED"] = (df_prev_app["NAME_CONTRACT_STATUS"] == "Refused").astype(int)
df_prev_app["CREDIT_TO_APP"] = df_prev_app["AMT_CREDIT"] / df_prev_app["AMT_APPLICATION"]
df_prev_app["CREDIT_TO_APP"] = df_prev_app["CREDIT_TO_APP"].replace([np.inf, -np.inf], np.nan)

In [42]:
agg_prev_app = df_prev_app.groupby("SK_ID_CURR").agg(
    PREV_COUNT=("SK_ID_PREV", "count"),
    PREV_REFUSED_COUNT=("IS_REFUSED", "sum"),
    PREV_REFUSED_SHARE=("IS_REFUSED", "mean"),
    PREV_DAYS_DECISION_MAX=("DAYS_DECISION", "max"),
    PREV_CREDIT_TO_APP_MEAN=("CREDIT_TO_APP", "mean"),
).reset_index()

agg_prev_app.describe().T[["mean", "min", "max"]]

,mean,min,max
SK_ID_CURR,278149.909581,100001.0,456255.00
PREV_COUNT,4.928964,1.0,77.00
PREV_REFUSED_COUNT,0.857819,0.0,68.00
PREV_REFUSED_SHARE,0.111420,0.0,1.00
PREV_DAYS_DECISION_MAX,-481.489404,-2922.0,-1.00
PREV_CREDIT_TO_APP_MEAN,1.018432,0.1,10.18


In [43]:
n_before = len(df2)
df3 = df2.merge(agg_prev_app, on="SK_ID_CURR", how="left")
assert len(df3) == n_before

In [44]:
df3.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,BUREAU_DEBT_SUM,BUREAU_OVERDUE_DAYS_MAX,BUREAU_OVERDUE_AMT_SUM,BUREAU_OVERDUE_COUNT,BUREAU_DEBT_RATIO,PREV_COUNT,PREV_REFUSED_COUNT,PREV_REFUSED_SHARE,PREV_DAYS_DECISION_MAX,PREV_CREDIT_TO_APP_MEAN
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,245781.0,0.0,0.0,0.0,0.284122,1.0,0.0,0.000000,-606.0,1.000000
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.0,0.0,0.0,0.0,0.000000,3.0,0.0,0.000000,-746.0,1.057664
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.0,0.0,0.0,0.0,0.000000,1.0,0.0,0.000000,-815.0,0.828021
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,NaN,NaN,NaN,0.0,NaN,9.0,1.0,0.111111,-181.0,1.012684
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.0,0.0,0.0,0.0,0.000000,6.0,0.0,0.000000,-374.0,1.046356


In [45]:
df3.info()

<class 'pandas.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 139 entries, SK_ID_CURR to PREV_CREDIT_TO_APP_MEAN
dtypes: float64(82), int64(41), str(16)
memory usage: 326.1 MB


In [46]:
count_cols3 = ["PREV_COUNT", "PREV_REFUSED_COUNT"]
df3[count_cols3] = df3[count_cols3].fillna(0)

In [47]:
y = df3["TARGET"]
X = df3.drop(columns=["TARGET", "SK_ID_CURR"])
cat_cols = X.select_dtypes("object").columns
X[cat_cols] = X[cat_cols].astype("category")

/var/folders/tg/m3kj7c9d5t9gsv408k8gwqkm0000gn/T/ipykernel_2514/3811546794.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes("object").columns


In [ ]:
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, verbose=-1)
model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="auc", callbacks=[lgb.early_stopping(200), lgb.log_evaluation(100)])

auc = roc_auc_score(y_va, model.predict_proba(X_va)[:, 1])
print(f"AUC = {auc:.4f}, Gini = {2 * auc - 1:.4f}")

/Users/daniil/projects/credit-scoring/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.763413	valid_0's binary_logloss: 0.244461
[200]	valid_0's auc: 0.767876	valid_0's binary_logloss: 0.242858
[300]	valid_0's auc: 0.768266	valid_0's binary_logloss: 0.242717
[400]	valid_0's auc: 0.768581	valid_0's binary_logloss: 0.242601
[500]	valid_0's auc: 0.768447	valid_0's binary_logloss: 0.242619
Early stopping, best iteration is:
[372]	valid_0's auc: 0.768742	valid_0's binary_logloss: 0.24257
AUC = 0.7687, Gini = 0.5375


In [49]:
imp = pd.Series(model.booster_.feature_importance(importance_type="gain"), index=X_tr.columns).sort_values(ascending=False)

imp.head(15)

EXT_SOURCE_3               63401.656285
EXT_SOURCE_2               53531.901140
ORGANIZATION_TYPE          23140.193594
EXT_SOURCE_1               20339.338355
PREV_CREDIT_TO_APP_MEAN    11267.671781
DAYS_EMPLOYED               9376.506756
BUREAU_DEBT_RATIO           8817.877171
AMT_CREDIT                  7207.453134
PREV_REFUSED_SHARE          7080.593529
BUREAU_DAYS_CREDIT_MAX      6550.119524
AMT_ANNUITY                 6440.165264
AMT_GOODS_PRICE             6348.056226
PREV_DAYS_DECISION_MAX      5176.226508
OCCUPATION_TYPE             4571.796058
OWN_CAR_AGE                 4362.031621
dtype: float64

In [76]:
df_inst_pay = pd.read_csv('data/installments_payments.csv')

df_inst_pay.head()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


In [77]:
df_inst_pay["DPD"] = (df_inst_pay["DAYS_ENTRY_PAYMENT"] - df_inst_pay["DAYS_INSTALMENT"]).clip(lower=0)
df_inst_pay["UNDERPAID"] = (df_inst_pay["AMT_INSTALMENT"] - df_inst_pay["AMT_PAYMENT"]).clip(lower=0)
df_inst_pay["IS_LATE"] = (df_inst_pay["DPD"] > 0).astype(int)
df_inst_pay[["DPD", "UNDERPAID", "IS_LATE"]].describe().T[["mean", "min", "max"]]

,mean,min,max
DPD,1.033984,0.0,2884.000
UNDERPAID,721.030023,0.0,2424726.405
IS_LATE,0.084280,0.0,1.000


In [78]:
agg_inst_pay = df_inst_pay.groupby("SK_ID_CURR").agg(
    INST_COUNT=("SK_ID_PREV", "count"),
    INST_PAID_COUNT=("AMT_PAYMENT", "count"),
    INST_DPD_MEAN=("DPD", "mean"),
    INST_DPD_MAX=("DPD", "max"),
    INST_LATE_SHARE=("IS_LATE", "mean"),
    INST_UNDERPAID_SUM=("UNDERPAID", "sum"),
    INST_PAYMENT_SUM=("AMT_PAYMENT", "sum"),
    INST_INSTALMENT_SUM=("AMT_INSTALMENT", "sum"),
).reset_index()

In [79]:
agg_inst_pay["INST_PAID_RATIO"] = (agg_inst_pay["INST_PAYMENT_SUM"] / agg_inst_pay["INST_INSTALMENT_SUM"]).replace([np.inf, -np.inf], np.nan)
agg_inst_pay["INST_UNPAID_COUNT"] = agg_inst_pay["INST_COUNT"] - agg_inst_pay["INST_PAID_COUNT"]

agg_inst_pay.describe().T[["mean", "min", "max"]]

,mean,min,max
SK_ID_CURR,278154.892278,100001.0,4.562550e+05
INST_COUNT,40.064552,1.0,3.720000e+02
INST_PAID_COUNT,40.055997,0.0,3.720000e+02
INST_DPD_MEAN,1.027695,0.0,1.885386e+03
INST_DPD_MAX,17.844109,0.0,2.884000e+03
INST_LATE_SHARE,0.074385,0.0,1.000000e+00
INST_UNDERPAID_SUM,28881.576754,0.0,3.037736e+06
INST_PAYMENT_SUM,690494.226221,0.0,3.268928e+07
INST_INSTALMENT_SUM,683136.948701,0.0,3.247978e+07
INST_PAID_RATIO,0.993382,0.0,5.485141e+00


In [80]:
n_before = len(df3)
df4 = df3.merge(agg_inst_pay, on="SK_ID_CURR", how="left")
assert len(df4) == n_before

In [81]:
df4.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,INST_COUNT,INST_PAID_COUNT,INST_DPD_MEAN,INST_DPD_MAX,INST_LATE_SHARE,INST_UNDERPAID_SUM,INST_PAYMENT_SUM,INST_INSTALMENT_SUM,INST_PAID_RATIO,INST_UNPAID_COUNT
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,19.0,19.0,0.000000,0.0,0.000000,0.000,219625.695,219625.695,1.000000,0.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,25.0,25.0,0.000000,0.0,0.000000,0.000,1618864.650,1618864.650,1.000000,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,3.0,3.0,0.000000,0.0,0.000000,0.000,21288.465,21288.465,1.000000,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,16.0,16.0,0.000000,0.0,0.000000,0.000,1007153.415,1007153.415,1.000000,0.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,66.0,66.0,0.954545,12.0,0.242424,29857.365,806127.975,835985.340,0.964285,0.0


In [82]:
count_cols4 = ["INST_COUNT", "INST_PAID_COUNT", "INST_UNPAID_COUNT"]
df4[count_cols4] = df4[count_cols4].fillna(0)

In [83]:
y = df4["TARGET"]
X = df4.drop(columns=["TARGET", "SK_ID_CURR"])
cat_cols = X.select_dtypes("object").columns
X[cat_cols] = X[cat_cols].astype("category")

/var/folders/tg/m3kj7c9d5t9gsv408k8gwqkm0000gn/T/ipykernel_2514/671355773.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes("object").columns


In [84]:
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, verbose=-1)
model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="auc", callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)])

auc = roc_auc_score(y_va, model.predict_proba(X_va)[:, 1])
print(f"AUC = {auc:.4f}, Gini = {2 * auc - 1:.4f}")

/Users/daniil/projects/credit-scoring/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[200]	valid_0's auc: 0.772765	valid_0's binary_logloss: 0.241476
[400]	valid_0's auc: 0.773546	valid_0's binary_logloss: 0.241221
Early stopping, best iteration is:
[385]	valid_0's auc: 0.773668	valid_0's binary_logloss: 0.241177
AUC = 0.7737, Gini = 0.5473


In [85]:
imp = pd.Series(model.booster_.feature_importance(importance_type="gain"), index=X_tr.columns).sort_values(ascending=False)

imp.head(15)

EXT_SOURCE_3               62275.625180
EXT_SOURCE_2               52728.526455
ORGANIZATION_TYPE          22915.017351
EXT_SOURCE_1               19580.483445
PREV_CREDIT_TO_APP_MEAN    10964.789847
DAYS_EMPLOYED               9730.292033
BUREAU_DEBT_RATIO           8600.548409
INST_LATE_SHARE             7613.134780
INST_PAYMENT_SUM            7537.409781
AMT_CREDIT                  7287.337951
BUREAU_DAYS_CREDIT_MAX      6470.931182
PREV_REFUSED_SHARE          6280.173871
AMT_ANNUITY                 6233.702586
AMT_GOODS_PRICE             5448.151992
OCCUPATION_TYPE             4555.500388
dtype: float64

In [86]:
%load_ext autoreload
%autoreload 2
from features import load_application, add_bureau, add_prev, add_installments, make_xy

new_df = add_installments(add_prev(add_bureau(load_application())))

/Users/daniil/projects/credit-scoring/features.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)


In [87]:
pd.testing.assert_frame_equal(df4.drop(columns="AGE"), new_df)